# 03 — Feature Engineering

**Milestone 3, Phase A + B.** Today's tickets: `HC-M3-02` (holdout strategy), `HC-M3-03` (experiment configuration). `HC-M3-01` (modeling objective) is doc-only — see [`docs/problem_definition.md`](../docs/problem_definition.md) §7.

**Notebook structure for Milestone 3** (deciding this now so it doesn't drift): three notebooks, one per pipeline stage, not one giant file —

- `03_feature_engineering.ipynb` (this one) — Phase A (foundation) + Phase B (feature engineering)
- `04_modeling.ipynb` — Phase C (preprocessing pipeline) through Phase G (final holdout evaluation)
- `05_error_analysis.ipynb` — Phase H

This is also where reusable logic starts moving out of notebooks and into `src/home_credit_default_risk/` as real, importable, testable modules — notebooks call into it rather than containing the logic inline. Per the instructor note this session is working from: Jupyter is sufficient, but scripts are preferred where there's time, and this is exactly the kind of reusable logic (feature functions, config, pipeline construction) that benefits from being real code with unit tests rather than notebook cells.

## `HC-M3-02` — Holdout strategy

**Decision: reuse the Milestone 1 split as the M3 dev/holdout boundary, rather than drawing a new one.** `data/interim/train_valid_split.csv` already is a stratified, seeded, reproducibility-verified 80/20 split (`HC-M1-05`, `HC-M1-09`). Relabeling it rather than resplitting means there's exactly one holdout definition across the whole project — a fresh split here would raise "why does this differ from Milestone 1" for no benefit, and would throw away an already-verified reproducibility guarantee.

Semantic relabeling for M3 (the underlying row assignment doesn't change):

- M1's **`train`** fold (246,008 rows) → M3's **development** pool — this is what `HC-M3-11`'s 5-fold CV runs inside, and what all feature engineering, preprocessing, and hyperparameter tuning ever touches.
- M1's **`valid`** fold (61,503 rows) → M3's **holdout** — frozen from this point forward. Not explored, not fit on, not used to pick between candidates or hyperparameters. Used exactly once, at `HC-M3-21`.

The config module (`src/home_credit_default_risk/config.py`, `HC-M3-03`) is used here rather than local constants, and the cell below doesn't just load the persisted split — it **recomputes** the split from `config.RANDOM_STATE`/`config.TEST_SIZE` against the current data and asserts the result is identical to what's on disk. That's a real reproducibility check, not a restated claim.

In [1]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split

from home_credit_default_risk import config

con = duckdb.connect(str(config.CACHE_DB), read_only=True)
application_train = con.sql("SELECT SK_ID_CURR, TARGET FROM application_train").df()
con.close()

ids = application_train["SK_ID_CURR"]
y = application_train["TARGET"]

print(f"RANDOM_STATE={config.RANDOM_STATE}, TEST_SIZE={config.TEST_SIZE}")

RANDOM_STATE=42, TEST_SIZE=0.2


In [2]:
recomputed_dev_ids, recomputed_holdout_ids = train_test_split(
    ids,
    test_size=config.TEST_SIZE,
    stratify=y,
    random_state=config.RANDOM_STATE,
)

persisted_split = pd.read_csv(config.SPLIT_PATH)
persisted_dev_ids = persisted_split.loc[
    persisted_split["split"] == "train", "SK_ID_CURR"
]
persisted_holdout_ids = persisted_split.loc[
    persisted_split["split"] == "valid", "SK_ID_CURR"
]

assert set(recomputed_dev_ids) == set(persisted_dev_ids), "Development set mismatch"
assert set(recomputed_holdout_ids) == set(persisted_holdout_ids), "Holdout set mismatch"

print("Recomputed split from config matches the persisted split exactly.")
print(f"Development pool: {len(persisted_dev_ids):,} rows")
print(f"Holdout:           {len(persisted_holdout_ids):,} rows")

Recomputed split from config matches the persisted split exactly.
Development pool: 246,008 rows
Holdout:           61,503 rows


**What's checked below is structural, not relational** — row counts and class balance per pool, which are mechanically guaranteed by `stratify=y` and already known before looking at any feature. This is not "peeking" at the holdout in the sense the project's EDA discipline forbids (feature-vs-target relationships, distributions that would inform imputation/outlier decisions) — it's confirming the split mechanism worked, the same category of check as `HC-M1-05`'s positive-rate confirmation.

In [3]:
y_by_id = y.set_axis(ids)
dev_positive_rate = y_by_id.loc[persisted_dev_ids].mean()
holdout_positive_rate = y_by_id.loc[persisted_holdout_ids].mean()

print(f"Development positive rate: {dev_positive_rate:.4f}")
print(f"Holdout positive rate:     {holdout_positive_rate:.4f}")

Development positive rate: 0.0807
Holdout positive rate:     0.0807


### Summary — HC-M3-02 acceptance criteria

- [x] Holdout split created (reused from `HC-M1-05`, relabeled — not redrawn)
- [x] Stratification applied (`stratify=y`, confirmed identical positive rate in both pools above)
- [x] Random state fixed (`config.RANDOM_STATE = 42`)
- [x] Holdout is never used during tuning (structural convention starting now: `persisted_holdout_ids` is not touched again until `HC-M3-21`)
- [x] Split is reproducible (recomputing from `config` against current data reproduced the persisted split exactly — verified above, not assumed)
- [x] Split strategy documented (this section)

Next: `HC-M3-03` — experiment configuration (already introduced above as `src/home_credit_default_risk/config.py`; closed out properly next).

## `HC-M3-03` — Experiment configuration

Already in use above, not introduced fresh here — `RANDOM_STATE`, `TEST_SIZE`, `N_SPLITS`, and `PRIMARY_METRIC` all live in [`src/home_credit_default_risk/config.py`](../src/home_credit_default_risk/config.py), a real importable module rather than notebook-local constants. That distinction matters for reproducibility going forward: `HC-M3-11`'s cross-validation, `HC-M3-16`/`17`'s hyperparameter search, and any script run outside a notebook all import the same `config`, so "which seed did we use" and "how many folds" have exactly one answer across the whole project, not one per notebook that can silently drift.

Covered by unit tests (`tests/test_config.py`) rather than just asserted — `RANDOM_STATE == 42`, `N_SPLITS == 5`, `PRIMARY_METRIC == "roc_auc"`, and that paths resolve under the project root, so a future edit that accidentally changes one of these fails CI/`pytest`, not silently drifts.

### Summary — HC-M3-03 acceptance criteria

- [x] Random seed centralized (`config.RANDOM_STATE`)
- [x] CV configuration centralized (`config.N_SPLITS`)
- [x] Evaluation metric centralized (`config.PRIMARY_METRIC`)
- [x] Configuration reproducible (unit-tested in `tests/test_config.py`; used and verified live in `HC-M3-02` above)

Next: `HC-M3-04` — feature engineering strategy.

## `HC-M3-04` — Feature engineering strategy

Doc-only, like `HC-M3-01` — see [`docs/feature_engineering_strategy.md`](../docs/feature_engineering_strategy.md). Six candidate feature groups, every source column verified against the real profiled schema, each with business rationale, leakage risk, and priority. Groups A (credit/income ratios) and B (age/employment) are implemented next, in `HC-M3-05`.

## `HC-M3-05` — Basic derived features

Implemented in [`src/home_credit_default_risk/features.py`](../src/home_credit_default_risk/features.py), not inline here — this is exactly the reusable, testable logic called out in the notebook-structure note at the top. Two things the acceptance criteria specifically call out, worth being precise about rather than hand-waving:

- **Division-by-zero and infinite values**: [`utils.safe_divide()`](../src/home_credit_default_risk/utils.py) maps a zero (or missing) denominator, and any resulting `+/-inf`, to `NaN`. Not because zero denominators exist in the current data (checked — they don't, `AMT_INCOME_TOTAL`/`AMT_ANNUITY`/`AMT_CREDIT` all have positive minimums in `application_train`), but because this function is meant to be reusable against `application_test` and any future data, and `+/-inf` breaks `SimpleImputer`/scalers downstream in a way `NaN` doesn't — the same category of problem as the `EMERGENCYSTATE_MODE` dtype bug from `HC-M1-07`, just prevented this time instead of discovered by a crash. Shared as a common helper (not duplicated) since `HC-M3-06`'s aggregation ratios need the exact same guarantee.
- **No target leakage by construction, not by convention**: every function takes only the feature dataframe, never `TARGET` — there's no code path through which the target could influence a derived feature, and `tests/test_features.py` asserts this explicitly rather than just relying on the function signature to make the point.

Applied and sanity-checked against the real development pool below (not just the synthetic unit tests).

In [4]:
import numpy as np

from home_credit_default_risk.features import add_basic_features

con = duckdb.connect(str(config.CACHE_DB), read_only=True)
application_train_full = con.sql("SELECT * FROM application_train").df()
con.close()

development = application_train_full[
    application_train_full["SK_ID_CURR"].isin(persisted_dev_ids)
].reset_index(drop=True)

development_with_features = add_basic_features(development)

new_columns = [
    "credit_to_income",
    "annuity_to_income",
    "credit_to_annuity",
    "goods_price_to_credit",
    "age_years",
    "employment_years",
    "is_employment_unknown",
    "employment_to_age_ratio",
]
development_with_features[new_columns].describe()

,credit_to_income,annuity_to_income,credit_to_annuity,goods_price_to_credit,age_years,employment_years,is_employment_unknown,employment_to_age_ratio
count,246008.000000,245998.000000,245998.000000,245787.000000,246008.000000,201865.000000,246008.000000,201865.000000
mean,3.959396,0.180911,21.624291,0.900686,43.886423,6.529758,0.179437,0.156934
std,2.687597,0.094571,7.820534,0.096646,11.944884,6.405822,0.383719,0.133635
min,0.004808,0.000224,8.036674,0.166667,20.503765,-0.000000,0.000000,-0.000000
25%,2.018667,0.114583,15.647004,0.834725,33.952088,2.097194,0.000000,0.056049
50%,3.268948,0.162833,20.000000,0.893815,43.104723,4.511978,0.000000,0.118734
75%,5.168514,0.229000,27.099985,1.000000,53.861739,8.695414,0.000000,0.219353
max,49.227200,1.570600,45.305079,6.666667,69.073238,49.040383,1.000000,0.728811


In [5]:
numeric_new = development_with_features[new_columns].select_dtypes(include="number")
inf_count = np.isinf(numeric_new.to_numpy()).sum()
unknown_employment_rate = development_with_features["is_employment_unknown"].mean()

sentinel_rows = development_with_features["is_employment_unknown"] == 1
sentinel_non_null = (
    development_with_features.loc[sentinel_rows, "employment_years"].notna().sum()
)

print(f"Infinite values across new columns: {inf_count}")
print(f"is_employment_unknown rate: {unknown_employment_rate:.4f}")
print(f"employment_years non-null for sentinel rows (should be 0): {sentinel_non_null}")

Infinite values across new columns: 0
is_employment_unknown rate: 0.1794
employment_years non-null for sentinel rows (should be 0): 0


### Summary — HC-M3-05 acceptance criteria

- [x] Features implemented in reusable code (`src/home_credit_default_risk/features.py`)
- [x] Missing values handled appropriately (propagate as `NaN`, left for the preprocessing pipeline's imputer — not silently filled here, which would bake in an assumption before modeling decides one)
- [x] Division-by-zero handled (`utils.safe_divide`, verified against real development data above: 0 infinite values)
- [x] Infinite values handled (same mechanism, same verification)
- [x] Unit tests added (`tests/test_features.py`, 9 tests, synthetic data covering zero/missing denominators, the sentinel, and the no-leakage/no-mutation guarantees; `tests/test_utils.py`, 4 tests, for the shared `safe_divide` helper)
- [x] No target leakage (functions never accept `TARGET`; asserted in tests, not just structurally implied)

Next: `HC-M3-06` — historical aggregation features (bureau, previous application, payment behaviour).

## `HC-M3-06` — Historical aggregation features

Implemented in [`src/home_credit_default_risk/aggregations.py`](../src/home_credit_default_risk/aggregations.py) — Groups C (bureau history), D (previous application behaviour), and E (payment behaviour) from the strategy doc. Aggregation runs as DuckDB SQL against the already-landed cache tables, the same pattern as `scripts/profile_data.py`, not pandas `.groupby()` on the full event tables (`bureau_balance` alone is 27M rows).

**Aggregation keys**: every source table groups by `SK_ID_CURR` directly, except `bureau_balance`, which has none of its own — grouped by `SK_ID_BUREAU`, joined through `bureau` to reach `SK_ID_CURR`, the same path documented in notebook 01.

**Missing history handled deliberately, not uniformly**: count-style features (`bureau_credit_count`, `prev_application_count`, ...) are filled with 0 for an applicant with no history — a true zero. Ratio/mean/max-style features (`bureau_debt_to_credit_ratio`, `mean_payment_ratio`, ...) are left as `NaN` — collapsing "no history" into "0" would fabricate a signal (a client with no previous credit isn't the same as one with a 0% debt ratio). This is the reusable module's own contract, verified in `tests/test_aggregations.py`, not restated here from memory.

**Row-level → application-level validation and leakage**: `build_historical_features()` is built around one non-negotiable check — `len(output) == len(spine_ids)`, since a fan-out (an accidental one-to-many join surviving into the result) would silently duplicate application rows, the single most damaging bug this kind of join can produce. Verified against real data below, and already covered against synthetic multi-row-per-key data in the unit tests (`test_build_historical_features_no_fanout_when_source_has_multiple_rows`). Leakage reasoning for each table is documented inline in `aggregations.py`'s docstrings; Group E (payment behaviour, keyed by `SK_ID_PREV`) is flagged there as the specific focus for the dedicated audit next, `HC-M3-07`.

In [6]:
from home_credit_default_risk.aggregations import build_historical_features

con = duckdb.connect(str(config.CACHE_DB), read_only=True)
historical_features = build_historical_features(con, development["SK_ID_CURR"])
con.close()

row_counts_match = len(historical_features) == len(development)
assert row_counts_match, (
    f"Row count mismatch: {len(historical_features)} vs {len(development)} "
    "-- a join fanned out."
)
assert historical_features["SK_ID_CURR"].is_unique, "Duplicate SK_ID_CURR after join"

print(f"historical_features: {historical_features.shape}")
print(f"Row count matches development pool exactly: {row_counts_match}")

historical_features: (246008, 22)
Row count matches development pool exactly: True


In [7]:
no_bureau_history_rate = (historical_features["bureau_credit_count"] == 0).mean()
no_prev_app_history_rate = (historical_features["prev_application_count"] == 0).mean()
no_installments_history_rate = historical_features["mean_payment_ratio"].isna().mean()

print(f"Applicants with no bureau history:       {no_bureau_history_rate:.4f}")
print(f"Applicants with no previous application: {no_prev_app_history_rate:.4f}")
print(f"Applicants with no installment history:   {no_installments_history_rate:.4f}")

Applicants with no bureau history:       0.1433
Applicants with no previous application: 0.0536
Applicants with no installment history:   0.0516


**14.33% of applicants have no bureau history at all** — worth noting against the business framing in `docs/problem_definition.md` §1: Home Credit's stated mission is serving clients with thin or no formal credit history, and this is a direct, measured confirmation of that at the portfolio level, not just a stated assumption. 5.36%/5.16% have no previous Home Credit application or installment history respectively — smaller gaps, consistent with a mix of new and returning clients.

### Summary — HC-M3-06 acceptance criteria

- [x] Aggregation keys documented (`SK_ID_CURR` direct; `bureau_balance` via `SK_ID_BUREAU → bureau`)
- [x] Aggregation logic implemented (`src/home_credit_default_risk/aggregations.py`, DuckDB SQL against landed tables)
- [x] Missing history handled (counts → 0, ratios/means/maxes → `NaN`; deliberate, not uniform — verified above and in tests)
- [x] Leakage checked (documented per table in `aggregations.py` docstrings; Group E flagged as `HC-M3-07`'s focus)
- [x] Row-level → application-level transformation validated (`len(output) == len(spine_ids)` asserted against real data, and against synthetic multi-row-per-key data in tests)
- [x] Output row count validated (246,008 in, 246,008 out — exact match, confirmed above)
- [x] Tests added (`tests/test_aggregations.py`, 7 tests against a synthetic in-memory DuckDB connection, every expected value hand-computed)

Next: `HC-M3-07` — feature leakage audit, focused on Group E (payment behaviour) as flagged throughout this section.